## load_silver_fhfa
Conforms `bronze.fhfa_hpi_master` into `silver.fact_fhfa_hpi_metro_quarterly`. Filters to the **canonical variant** — `level='MSA' AND hpi_type='traditional' AND hpi_flavor='all-transactions'` (1975+, NSA-only). Joins `dim_geo` directly on `place_id` = `cbsa_code`.

**Transforms:** `index_nsa` and `rstderr`→`standard_error` cast to DOUBLE (kept fractional — index points); `date_key` from `yr` + `period` (quarter) → quarter-end. Unmatched CBSA or malformed value → `silver.quarantine`. MERGE on `(geo_key, date_key)`; StepLog + transform_detail_log.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 1
SOURCE_SYSTEM = "fhfa"
SOURCE_TABLE  = f"{BRONZE}.fhfa_hpi_master"
TARGET_TABLE  = f"{SILVER}.fact_fhfa_hpi_metro_quarterly"
QUARANTINE    = f"{SILVER}.quarantine"
DIM_GEO       = f"{SILVER}.dim_geo"
VALUE_COLS    = ["index_nsa", "standard_error"]   # both DOUBLE (index points, not USD)

In [ ]:
# Open the pipeline_step_log row (RUNNING).
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "silver",
    target_table    = TARGET_TABLE,
)
print(f"load_silver_fhfa: step_log_id={step.step_log_id}")

In [ ]:
# Read + filter to the canonical variant, type the two metrics, derive the quarter-end
# date_key, detect cast errors, capture raw_payload, and resolve geo_key on place_id.
try:
    bronze = spark.table(SOURCE_TABLE).where(
        "level = 'MSA' AND hpi_type = 'traditional' AND hpi_flavor = 'all-transactions'")
    rows_read = bronze.count()

    # yr + period(1-4) -> quarter-end date (month = period*3) -> yyyymmdd INT.
    q_end = F.last_day(F.make_date(F.col("yr").cast("int"),
                                   F.col("period").cast("int") * 3, F.lit(1)))
    date_key = (F.year(q_end) * 10000 + F.month(q_end) * 100 + F.dayofmonth(q_end)).cast("int")

    cast_err = lambda c: (F.col(c).isNotNull()) & (F.trim(F.col(c)) != F.lit("")) \
                         & (F.col(c).cast("double").isNull())
    typed = bronze.select(
        F.col("place_id").alias("cbsa_code"),
        F.col("yr"), F.col("period"),
        date_key.alias("date_key"),
        F.col("index_nsa").cast("double").alias("index_nsa"),
        F.col("rstderr").cast("double").alias("standard_error"),
        F.array_compact(F.array(
            F.when(cast_err("index_nsa"), F.lit("index_nsa")),
            F.when(cast_err("rstderr"),   F.lit("standard_error")),
        )).alias("cast_errors"),
        F.to_json(F.struct(*[F.col(x) for x in bronze.columns])).alias("raw_payload"),
    )
    geo = spark.table(DIM_GEO).select("geo_key", "cbsa_code")
    staged = typed.join(geo, "cbsa_code", "left")
    step.rows_read = rows_read
    print(f"load_silver_fhfa: read {rows_read:,} all-transactions MSA rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Split, quarantine (idempotent), MERGE on (geo_key, date_key), audit. §11.4 vars pre-declared.
transform_started = datetime.now(timezone.utc)
rows_rejected = rows_inserted = rows_updated = 0
try:
    good = staged.where((F.col("geo_key").isNotNull()) & (F.size("cast_errors") == 0))
    bad  = staged.where((F.col("geo_key").isNull()) | (F.size("cast_errors") > 0))
    rows_rejected = bad.count()

    spark.sql(f"DELETE FROM {QUARANTINE} WHERE source_system = '{SOURCE_SYSTEM}'")
    if rows_rejected > 0:
        reason = F.when(F.col("geo_key").isNull(), F.lit("unmatched_geography")) \
                  .otherwise(F.concat(F.lit("cast_failed:"), F.concat_ws(",", F.col("cast_errors"))))
        bad.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.lit(None).cast("string").alias("source_file_path"),
            F.concat_ws("|", F.col("cbsa_code"), F.col("yr"), F.col("period")).alias("natural_key"),
            F.col("raw_payload"),
            reason.alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)

    fact_cols = ["geo_key", "date_key"] + VALUE_COLS
    good.select(*[F.col(c) for c in fact_cols],
                F.current_timestamp().alias("inserted_ts"),
                F.current_timestamp().alias("updated_ts")).createOrReplaceTempView("fhfa_fact_staging")

    set_clause = ", ".join(f"t.{c}=s.{c}" for c in VALUE_COLS) + ", t.updated_ts=s.updated_ts"
    cols_csv   = ", ".join(fact_cols + ["inserted_ts", "updated_ts"])
    vals_csv   = ", ".join(f"s.{c}" for c in fact_cols + ["inserted_ts", "updated_ts"])
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} t USING fhfa_fact_staging s
        ON t.geo_key = s.geo_key AND t.date_key = s.date_key
        WHEN MATCHED THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT ({cols_csv}) VALUES ({vals_csv})
    """).first().asDict()
    rows_inserted = metrics.get("num_inserted_rows") or 0
    rows_updated  = metrics.get("num_updated_rows") or 0

    step.rows_written = rows_inserted
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=rows_inserted, rows_inserted=rows_inserted, rows_updated=rows_updated,
        rows_rejected=rows_rejected, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"load_silver_fhfa: inserted={rows_inserted:,} updated={rows_updated:,} "
          f"quarantined={rows_rejected:,} (read={step.rows_read:,})")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_rejected=rows_rejected, error_message=f"{type(e).__name__}: {e}",
        ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise